# LifeOps — A Privacy-First Personal Concierge (Concierge Agents track)

**Capstone — 5-Day AI Agents: Intensive Vibe Coding Course with Google**

## Problem
Everyone wants an assistant that "just handles" email and calendar — but adoption is gated on **trust**. The moment an agent can send mail or accept invites on your behalf, a hallucination or a prompt-injection email becomes a real-world action.

## Solution
LifeOps is a multi-agent concierge where **no external action ever happens without passing a single safety chokepoint**: a Coordinator routes to Email and Calendar specialists that can only *propose* actions; every proposal runs through a PII filter and a deny-by-default **Policy Server**, and anything risky waits for **human approval** before it executes.

## Architecture
```
Trigger (you ask  OR  ambient event)
      |
  Coordinator --A2A--> Specialist (Email / Calendar)   <- hold no write power
      |                      | proposes a world-changing action
      |                      v
      |               === Action Bus (single chokepoint) ===
      |               PII filter -> Policy Server verdict
      |                 allow ............ execute now
      |                 require_approval .. queue -> human approves -> execute
      |                 deny ............. blocked, logged
      v
  result + OpenTelemetry audit span on every decision
```

## Course concepts demonstrated
| Concept | In this project |
|---|---|
| Agent / multi-agent (ADK) | Coordinator + Email + Calendar sub-agents |
| MCP Server | Gmail/Calendar client interface + selector (`LIFEOPS_USE_MCP`); real calls wired in `agents-cli playground` |
| Security features | Policy Server, Action Bus chokepoint, PII filter, human-in-the-loop, credential-free Coordinator |
| Deployability | `agents-cli` manifest + FastAPI app + Dockerfile |

### How to run this notebook
1. Settings -> **Internet: On** (needed to clone + install).
2. The **safety demo + tests run with no credentials**.
3. *(Optional)* For the live Gemini path, add a **Kaggle Secret** named `GOOGLE_API_KEY` (a Google AI Studio key) via *Add-ons -> Secrets*. Never paste the key into a cell.

## Setup — clone the repo and install

In [ ]:
import os

REPO_URL = "https://github.com/soheil-aa/lifeops.git"  # public capstone repo

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
dst = os.path.join(WORK, "lifeops")
if not os.path.isdir(dst):
    !git clone -q $REPO_URL "$dst"
%cd "$dst"
!ls

In [ ]:
# Install with uv. pip's resolver backtracks for a long time against Kaggle's
# large pre-installed stack; uv resolves the same tree in seconds.
# (Internet must be On.)
!pip install -q uv
!uv pip install --system -q -e . pytest pytest-asyncio

## 1. The safety core — runs offline, no credentials
This drives the **real specialist tools** through the Action Bus with fake email/calendar clients. Watch `email.sent`: it stays at 0 until you explicitly approve, strangers and PII-leaks are denied, and a prompt-injection email body cannot make the agent auto-send.

In [ ]:
!python examples/demo.py

## 2. The test suite — the guarantees are proven, not asserted
Includes `test_safety_invariant.py`: across known/stranger/injection/PII/oversized cases, **no send ever executes during `propose()`**.

In [ ]:
!python -m pytest tests/unit tests/integration -q

## 3. Live Gemini path *(optional)* — real multi-agent routing
Runs only if a `GOOGLE_API_KEY` Kaggle Secret is set. It drives the real Coordinator through ADK's `Runner` (still using safe fake email/calendar). If no key is present, it skips gracefully — the offline demo above already proves the safety core.

In [ ]:
import os

HAVE_KEY = False
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
    HAVE_KEY = True
except Exception:
    print("No GOOGLE_API_KEY secret found -> skipping live run.")
    print("Add one via Add-ons -> Secrets to enable it. The offline demo already proves the safety core.")

if HAVE_KEY:
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.adk.agents.run_config import RunConfig, StreamingMode
    from google.genai import types
    from app.agent import root_agent  # import-safe; uses fake clients by default

    session_service = InMemorySessionService()
    session = session_service.create_session_sync(user_id="demo", app_name="lifeops")
    runner = Runner(agent=root_agent, session_service=session_service, app_name="lifeops")
    msg = types.Content(role="user", parts=[types.Part.from_text(
        text=("Draft a short reply to sarah@example.com confirming Friday lunch, "
              "then tell me whether sending it needs my approval and why."))])
    try:
        for event in runner.run(new_message=msg, user_id="demo", session_id=session.id,
                                run_config=RunConfig(streaming_mode=StreamingMode.SSE)):
            if event.content and event.content.parts:
                for p in event.content.parts:
                    if getattr(p, "text", None):
                        print(p.text, end="")
        print()
    except Exception as e:
        print(f"Live run error (model/quota/network): {e}")

## Limitations & roadmap (honest v1 scope)
- **Fake clients by default.** The Gmail/Calendar MCP clients are interface stubs in v1; real calls are wired during `agents-cli playground`. The safety architecture is fully real and runs offline.
- **Token Broker** is built and unit-tested as the JIT least-privilege seam, but not yet threaded into the live execute path. In v1 the Confused-Deputy defense comes from the architecture: the Coordinator holds no credentials, specialists can only propose, and the chokepoint enforces deny-by-default + human approval.
- **Next (v2 = "real tool"):** wire the Token Broker + real Gmail/Calendar MCP; add a semantic (LLM-as-judge) policy layer for fuzzy cases.

**Repo:** https://github.com/soheil-aa/lifeops &nbsp;|&nbsp; **Track:** Concierge Agents &nbsp;|&nbsp; no API keys or secrets are committed in this project.